データソース
気象：気象庁  
https://www.data.jma.go.jp/risk/obsdl/index.php?utm_source=chatgpt.com
＊一回にダウンロードできる量に制限がある。2021年1月から、各都道府県で3ヶ月単位でダウンロードする。
*代表地点として都道府県内の地方気象台を選択→日射量や天気、雲量などの情報があり、情報が多いため。
*格納されているデータ、前1時間。1時間前！時間をずらさないとだめ。
*本来の需要予測は予測値を使うが、今回は実績値の公開データで代用する。

電力需要：東京電力パワーグリッド  
https://www.tepco.co.jp/forecast/html/area_data-j.html

In [ ]:
import glob
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

1 データ読み込み

1.1 JMA(気象庁データ)

In [ ]:
# 東電管内の都道府県
# 静岡は半分中部なので除外
list_pref_names = ["chiba", "gumma", "ibaraki", "kanagawa", "saitama", "tochigi", "tokyo", "yamanashi"]

In [ ]:
# 元ファイルのヘッダーの英語変換。ただし、現象有無フラグ?があり、ヘッダーの名称が被っているからdictなどで１対１対応ができない。

# 気温(℃)：Temperature 降水量(mm)：Precipitation 降雪(cm)：Snowfall 積雪 ：Snow Depth 
# 日照時間(時間)：Sunshine Duration 風速(m/s)：Wind Speed 露点温度(℃)：Dew Point Temperature 
# 蒸気圧(hPa)：Vapor Pressure 相対湿度(％)：Relative Humidity 日射量(MJ/㎡)：Solar Radiation 
# 現地気圧(hPa)：Local Pressure 海面気圧(hPa)：Sea Level Pressure 天気：Weather 雲量(10分比)：Cloud Cover 
# 視程(km)：Visibility (km)


In [ ]:
# 気象庁のデータを読み込むための設定
list_columns_jma = [
    "datetime","temperature", "precipitation", "precipitation_flag", 
    "snowfall", "snowfall_flag", "snow_depth", "snow_depth_flag", 
    "sunshine_duration", "sunshine_duration_flag", "wind_speed", "wind_direction", 
    "dew_point", "vapor_pressure", "humidity", "solar_radiation", 
    "local_pressure", "sea_level_pressure", "weather", "cloud_cover", "visibility"
]

encoding_jma = "cp932"

dir_path_jma = "../data/raw/jma"

In [ ]:
len(list_columns_jma)

In [ ]:
# 気象庁のデータ読み込んでみる

df_jma = pd.read_csv("../data/raw/jma/tokyo/2021_01010331.csv", 
                          skiprows=6, 
                          names=list_columns_jma, 
                          header=None, 
                          encoding=encoding_jma)

In [ ]:
df_jma.head(5)

In [ ]:
df_jma["datetime"] = pd.to_datetime(df_jma["datetime"])

In [ ]:
df_jma

In [ ]:
# ファイルを結合する

df_weather = pd.DataFrame()

for pref in list_pref_names:
    dir_path = os.path.join(dir_path_jma,pref)

    # 都道府県ごとにファイル一覧を取得
    list_file_names = glob.glob(os.path.join(dir_path,"*.csv"))

    # csv->dfした元ファイルを一時的に格納
    dfs = []

    # 特徴量名が重複しないように都道府県名をprefixにする
    # datetimeはmergeのkeyにするため、そのまま
    list_col_names = [f"{pref}_{col}" if col != "datetime" else col for col in list_columns_jma ]

    # print(list_col_names)

    for path in list_file_names:
        df = pd.read_csv(path, 
                          skiprows=6, 
                          names=list_col_names, 
                          header=None, 
                          encoding=encoding_jma)
        # df["pref"] = pref
        dfs.append(df)
        
    df_concat = pd.concat(dfs)
    df_concat["datetime"] = pd.to_datetime(df_concat["datetime"])
    df_concat.sort_values(by="datetime", inplace=True)

    if df_weather.empty:
        df_weather = df_concat
    else:
        df_weather = pd.merge(df_weather, df_concat, on="datetime")

    # print(list_file_names)
    # break

In [ ]:
df_weather.head(5)

In [ ]:
# 気象庁データは1時間のずれがあるので、補正する。
# 0時のデータが1時として入っている
df_weather["datetime"] = df_weather["datetime"] - pd.Timedelta(hours=1)

In [ ]:
# 時間が-1できているか確認
df_weather.head(5)

In [ ]:
# マージした気象データを保存
df_weather.to_csv("../data/processed/jma/raw.csv", encoding="utf-8", index=False)

1.2 東電の電力需要データを読み込む

In [ ]:
# 近年のデータは月毎、数年前のデータは年度ごとにデータが格納されている
# 別々に読み込む

dir_path_tepco = "../data/raw/tepco"
encoding_tepco = "cp932"
list_columns_tepco = ["date", "time", "demand"]

1.2-1年ごとに１ファイルになっている年度のデータ

In [ ]:
# 年ごとのデータの確認

df_demand_yearly = pd.read_csv(os.path.join(dir_path_tepco, "yearly","area-2021.csv"), 
                               skiprows=3, 
                               usecols=range(0,3),
                               names=list_columns_tepco, 
                               header=None, 
                               encoding=encoding_tepco
                               )

In [ ]:
df_demand_yearly.head(5)

In [ ]:
# 年ごとに１ファイルになっているファイルを合算する
list_file_names_yearly = glob.glob(os.path.join(dir_path_tepco, "yearly","*.csv"))

In [ ]:
# 複数ファイル合算するように初期化
df_demand_yearly = pd.DataFrame() 

dfs = []

for path in list_file_names_yearly:
    df = pd.read_csv(path, 
                    skiprows=3, 
                    usecols=range(0,3),
                    names=list_columns_tepco, 
                    header=None, 
                    encoding=encoding_tepco
                    )
    
    dfs.append(df)

df_demand_yearly = pd.concat(dfs)
df_demand_yearly["datetime"] = df_demand_yearly["date"].str.cat(df_demand_yearly["time"], sep=" ")
df_demand_yearly["datetime"] = pd.to_datetime(df_demand_yearly["datetime"], format="%Y/%m/%d %H:%M")
df_demand_yearly.sort_values(by="datetime", inplace=True)
df_demand_yearly = df_demand_yearly.reset_index(drop=True)


In [ ]:
df_demand_yearly.head(5)

In [ ]:
# yearlyの需要の単位が万kwで、monthlyの単位がMWだから、yearlyのdemandに10をかけて、単位をMWに合わせる
df_demand_yearly["demand"] = df_demand_yearly["demand"] * 10

1.2-2月ごとに１ファイルになっている年度のデータ

In [ ]:
# 月毎のデータ
# １ファイルで中身確認

df_demand_monthly = pd.read_csv(os.path.join(dir_path_tepco, "monthly","eria_jukyu_202402_03.csv"), 
                               skiprows=2, 
                               usecols=range(0,3),
                               names=list_columns_tepco, 
                               header=None, 
                               )



In [ ]:
df_demand_monthly.head(5)

In [ ]:
# 月ごとに１ファイルになっているファイルを合算する
list_file_names_monthly = glob.glob(os.path.join(dir_path_tepco, "monthly","*.csv"))

In [ ]:
# 複数ファイル合算するように初期化
df_demand_monthly = pd.DataFrame() 

dfs = []

for path in list_file_names_monthly:
    df = pd.read_csv(path, 
                    skiprows=2, 
                    usecols=range(0,3),
                    names=list_columns_tepco, 
                    header=None, 
                    )
    
    dfs.append(df)

df_demand_monthly = pd.concat(dfs)
df_demand_monthly["datetime"] = df_demand_monthly["date"].str.cat(df_demand_monthly["time"], sep=" ")
df_demand_monthly["datetime"] = pd.to_datetime(df_demand_monthly["datetime"], format="%Y/%m/%d %H:%M")
df_demand_monthly.sort_values(by="datetime", inplace=True)
df_demand_monthly = df_demand_monthly.reset_index(drop=True)

In [ ]:
df_demand_monthly.head(5)

In [ ]:
# 気象庁の気象データと、東電の年ごとにまとめられている電力需要データは1時間間隔
# 月ごとにまとめられたデータは30分間隔なので、平均を取り簡易的に1時間に変換。

df_demand_monthly_resampled = df_demand_monthly[["datetime","demand"]].set_index("datetime").resample("h").mean()

In [ ]:
df_demand_monthly_resampled

1.2-3 年ごと、月ごと、ファイルの合算

In [ ]:
df_demand = pd.concat([df_demand_yearly[["datetime","demand"]].set_index("datetime"), 
                       df_demand_monthly_resampled])

In [ ]:
df_demand.head(5)

In [ ]:
# 年毎、月毎のマージにともなう重複の確認
df_demand.index.is_unique

In [ ]:
# 年毎のデータの方が古く、リサンプリングの処理を入れてないため、
# 重複削除はkeep=firstとする

df_demand_unique = df_demand[~df_demand.index.duplicated(keep="first")]

In [ ]:
df_demand_unique

In [ ]:
df_demand_unique.to_csv("../data/processed/tepco/raw.csv", encoding="utf-8", index=False)

2 気象データと電力需要データをマージ

In [ ]:
df_weather.head(5)

In [ ]:
df_demand_unique.head(5)

In [ ]:
# 学習データは2021年1月1日~2025年12月31日
# 電力需要データには、2021年1月1日以前のデータが入っているが、inner joinで該当期間を除外する

df_raw = pd.merge(df_weather, 
                  df_demand_unique.reset_index(), 
                  on="datetime", 
                  how="inner")

In [ ]:
# 始まりと終わりが、2021-01-01 00:00:00, 2025-12-31 23:00:00になっているか確認
df_raw

In [ ]:
df_raw.to_csv("../data/processed/jma_tepco_merged/raw.csv", encoding="utf-8", index=False)

3. データの確認

3.1基礎統計量の確認

In [ ]:
df_raw.describe()

3.2欠損値の確認

In [ ]:
# 電力需要の欠損値の確認
print(df_raw["demand"].isnull().sum())

In [ ]:
# 気象データのカラム名の再掲
list_columns_jma

In [ ]:
# 気温の欠損値の確認
df_raw.filter(like="temperature").isnull().sum()

In [ ]:
# 降水量の欠損値の確認
df_raw.filter(like="precipitation").isnull().sum()

In [ ]:
# 降雪の欠損値の確認
df_raw.filter(like="snowfall").isnull().sum()

#欠損値が多いから、補正とかではなく除外する

In [ ]:
# 積雪の深さの欠損値の確認
df_raw.filter(like="snow_depth").isnull().sum()

#欠損値が多いから、補正とかではなく除外する

In [ ]:
# 日照時間の欠損値の確認
df_raw.filter(like="sunshine_duration").isnull().sum()


In [ ]:
# 風速の欠損値の確認
df_raw.filter(like="wind_speed").isnull().sum()

In [ ]:
# 風向の欠損値の確認
df_raw.filter(like="wind_direction").isnull().sum()

In [ ]:
# 露点の欠損値の確認
df_raw.filter(like="dew_point").isnull().sum()

In [ ]:
# 蒸気圧の欠損値の確認
df_raw.filter(like="vapor_pressure").isnull().sum()

In [ ]:
# 湿度の欠損値の確認
df_raw.filter(like="humidity").isnull().sum()

In [ ]:
# 日射量の欠損値の確認
df_raw.filter(like="solar_radiation").isnull().sum()

# 欠損値の個数の差が大きい。
# 補正ではなく、除外する

In [ ]:
# 現地気圧の欠損値の確認
df_raw.filter(like="local_pressure").isnull().sum()

In [ ]:
# 海面気圧の欠損値の確認
df_raw.filter(like="sea_level_pressure").isnull().sum()

In [ ]:
# 天気の欠損値の確認
df_raw.filter(like="weather").isnull().sum()

# 東京だけ欠損値が大きい
# 一旦特徴量としては東京以外を入れてみて、精度を比較する

In [ ]:
# 雲量(10分比)の欠損値の確認
df_raw.filter(like="cloud_cover").isnull().sum()

# ほとんど欠損値だから除外する

In [ ]:
# 視程(km)の欠損値の確認
df_raw.filter(like="visibility").isnull().sum()

# 東京だけ欠損値が大きい
# 一旦特徴量としては東京以外を入れてみて、精度を比較する


3.3質的変数の表記ブレの確認

In [ ]:
# 気象データのうち、質的変数かつ欠損値が多くないカラム：
# weather, wind_direction

for col_name in df_raw.filter(like="weather").columns:
    print(col_name, df_raw[col_name].unique())

In [ ]:
# 気象庁の天気とラベル
dict_jma_weather = {
    1: "快晴",
    2: "晴れ",
    3: "薄曇",
    4: "曇",
    5: "煙霧",
    6: "砂じん嵐",
    7: "地ふぶき",
    8: "霧",
    9: "霧雨",
    10: "雨",
    11: "みぞれ",
    12: "雪",
    13: "あられ",
    14: "ひょう",
    15: "雷",
    16: "しゅう雨または止み間のある雨",
    17: "着氷性の雨",
    18: "着氷性の霧雨",
    19: "しゅう雪または止み間のある雪",
    22: "霧雪",
    23: "凍雨",
    24: "細氷",
    28: "もや",
    101: "降水またはしゅう雨性の降水"
}

In [ ]:
# wind_direction

for col_name in df_raw.filter(like="wind_direction").columns:
    print(col_name, df_raw[col_name].unique())
    # print(col_name, len(df_raw[col_name].unique()))
    
    break

4データの中身の確認

In [ ]:
df_eda = df_raw.copy()

4.1 欠損値の処理

In [ ]:
# カラムの数を確認
df_eda.columns

In [ ]:
# 欠損値が多いカラムを削除
list_remove_col_name =["cloud_cover", "solar_radiation", "snow_depth", "snowfall"]

for col in list_remove_col_name:
    df_eda = df_eda.drop(df_eda.filter(like=col).columns, axis=1)
    
    # break

In [ ]:
# カラムの数を確認
df_eda.columns

4.2値の分布

In [ ]:
# グラフ作成時に時間と質的変数は一旦除外
# list_temporary_drop = ["datetime"]
# list_temporary_drop.extend(df_eda.filter(like="weather").columns)
# list_temporary_drop.extend(df_eda.filter(like="wind_direction").columns)



In [ ]:
# 時系列で見てみる
x = df_eda["datetime"]
y = df_eda["demand"]

plt.scatter(x, y)
plt.show()

In [ ]:
# month vs demand(average)
df_eda["month"] = df_eda["datetime"].dt.month

In [ ]:
data = df_eda[["month","demand"]].groupby("month").mean()

plt.plot(data)

plt.xlabel("month")
plt.ylabel("demand")
plt.show()


In [ ]:
# hour vs demand
df_eda["hour"] = df_eda["datetime"].dt.hour

In [ ]:
data = df_eda[["hour","demand"]].groupby("hour").mean()

plt.plot(data)

plt.xlabel("hour")
plt.ylabel("demand")
plt.show()

In [ ]:
# temperature(小数点きりすて) vs demand

df_eda["temperature_average"] = df_eda.filter(like="temperature").mean(axis=1)

In [ ]:
# 型の確認
type(df_eda["temperature_average"][0])

In [ ]:
x = df_eda["temperature_average"].astype(int)
y = df_eda["demand"].astype(int)

plt.scatter(x, y)
# plt.plot(x, y)

plt.xlabel("tempareture")
plt.ylabel("demand")
plt.show()

In [ ]:
# 15~20度の過ごしやすい気温帯は電力需要が減る
# 気温の影響が大きい可能性

In [ ]:
df_eda.head(5)

In [ ]:
# 月毎に、気温の平均vs需要
for month in df_eda["month"].unique():

    df = df_eda[df_eda["month"]==month]

    x = df["temperature_average"].astype(int)
    y = df["demand"].astype(int)

    plt.scatter(x, y, label=month)

    # break
plt.xlabel("temperature_average")
plt.ylabel("demand")
plt.show()

In [ ]:
# 時間毎に、気温の平均vs需要

for hour in df_eda["hour"].unique():

    df = df_eda[df_eda["hour"]==hour]

    x = df["temperature_average"].astype(int)
    y = df["demand"].astype(int)

    plt.scatter(x, y, label=hour)

    # break
plt.xlabel("temperature_average")
plt.ylabel("demand")
plt.show()

In [ ]:
# wind_speed(小数点切り捨て) vs demand

df_eda["wind_speed_average"] = df_eda.filter(like="wind_speed").mean(axis=1)

In [ ]:
x = df_eda["wind_speed_average"].astype(int)
y = df_eda["demand"].astype(int)

plt.scatter(x, y)
# plt.plot(x, y)

plt.xlabel("wind_speed_average")
plt.ylabel("demand")
plt.show()

In [ ]:
# →wind speedはあまり傾向がない

In [ ]:
# humidity(小数点きりすて) vs demand
df_eda["humidity_average"] = df_eda.filter(like="humidity").mean(axis=1)


In [ ]:
# 湿度　vs　電力需要
x = df_eda["humidity_average"].astype(int)
y = df_eda["demand"].astype(int)

plt.scatter(x, y)
# plt.plot(x, y)

plt.xlabel("humidity_average")
plt.ylabel("demand")
plt.show()

In [ ]:
# 湿度は、思ったより強い相関はなさそう？

In [ ]:
# weather の分布
# df_eda["humidity_average"] = 
s = df_eda.filter(like="weather").value_counts().to_frame()

x = s.index.get_level_values(level=0)
y = s["count"]

plt.bar(x, y)
plt.xlabel("weather")
plt.ylabel("count")
plt.show()
# .mean(axis=1)


4.3相関係数（＊変数が多いから一旦保留）

In [ ]:
corr_matrix = df_eda.drop(list_temporary_drop, axis=1).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.show()

# 変数が多くて潰れる。
# 相関係数出すなら、気温とかごとに、平均とる
# でも高次元の相関係数を見ても・・・わからんから全体のヒートマップなくていいか。